<a href="https://colab.research.google.com/github/ecloguehwang/2026-Seojinhyeop-Workshop/blob/main/1%EA%B0%9C%ED%95%A9%EB%B6%88%ED%8C%8C%EC%9D%BC_1%EA%B0%9C%EC%8B%9C%ED%8A%B8_%EC%A0%84%EC%B2%98%EB%A6%AC_E%EA%B3%A02023%ED%95%99%EB%85%84%EB%8F%84.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 구글 드라이브의 Colab에서 작업하는 경우
## 3개의 폴더를 만들어 놓기

1. 학교명(ex: 금천고) 폴더 - 작업 폴더
2. 합불자료_전처리: 전처리 폴더
3. python: 데이터 폴더

### 4. 학교명 폴더 코드 실행시
#### 4-1 다음 코드 실행

#####colab에서 matplotlib와 sns 라이브러리 그래프 한글을 깨지지 않게 하는 법A: 이 코드실행하고 런타임(runtime) 다시 실행하기
<코드>
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

#####


#1개 합불파일에 1개(수시)의 시트가 있을때 새로운 합불파일로 만들기

## 주의점1: 실행하기 전에 원본파일의 열번호를 확인후 실행
## 주의점2: df.columns[32:36]: 33번째열부터 36번열까지를 의미!
## 주의점3: 원본의 첫번째 행은 인덱스로 자동 인식:
따라서 원본의 세번째 행을 코드에서는 두번째 행[1]으로 표시함!



In [ ]:
#1단계 - 1개 파일에 1개의 시트(수시)를 하나로 합치기


import pandas as pd

# 엑셀 파일 경로 설정
path = '/content/drive/MyDrive/python/habbul/'
df_susi = pd.read_excel(f'{path}susi_eunpyeong_2023.xlsx')


# 수시합불자료 데이터 클리닝
df_susi.columns = df_susi.iloc[1]  # 두 번째 행을 열 이름으로 설정
df_susi = df_susi.drop([0, 1])      # 첫 두 행 삭제
df_susi.reset_index(drop=True, inplace=True) # 인덱스 재설정



##5. 수시합불자료의 열 이름 변경
df_susi.columns.values[32:36] = ['국어_' + col for col in df_susi.columns[32:36]]
df_susi.columns.values[36:40] = ['수학_' + col for col in df_susi.columns[36:40]]
df_susi.columns.values[40:41] = ['영어_' + col for col in df_susi.columns[40:41]]
df_susi.columns.values[42:46] = ['선택1_' + col for col in df_susi.columns[42:46]]
df_susi.columns.values[46:50] = ['선택2_' + col for col in df_susi.columns[46:50]]
df_susi.columns.values[50:51] = ['한국사_' + col for col in df_susi.columns[50:51]]
df_susi.columns.values[51:53] = ['제2외국어_' + col for col in df_susi.columns[51:53]]


# 열 이름에서 \n 삭제
df_susi.columns = df_susi.columns.str.replace('\n', '', regex=False)


# 결과 DataFrame 출력
print(df_susi.tail())

In [ ]:
#2단계: 수시, 정시 파일합치고 상자그림용 문자열 제작

import pandas as pd


# Reset the index
df_susi.reset_index(drop=True, inplace=True)


# 전형분류 열 값 변경A: 수능 전문대 전형분류를 단순화
df_susi.loc[df_susi['전형분류'] == '면접위주', '전형분류'] = '종합'
df_susi.loc[df_susi['전형분류'] == '실기위주', '전형분류'] = '실기'
df_susi.loc[df_susi['전형분류'] == '학생부위주', '전형분류'] = '교과'


# 전형분류 열 값 변경B: 수능 전문대 전형분류를 단순화
df_susi['전형분류'] = df_susi['전형분류'].replace({'수능위주': '수능',
                                                       '실기/실적위주': '실기',
                                                       '일반전형': '실기'})




# 열 데이터를 숫자형으로 변환
cols_to_convert = ['국어_백분위', '수학_백분위', '선택1_백분위', '선택2_백분위',
                   '국어_표준점수', '수학_표준점수', '선택1_표준점수', '선택2_표준점수',
                   '국어_등급', '수학_등급', '영어_등급', '선택1_등급', '선택2_등급']


for col in cols_to_convert:
    df_susi[col] = pd.to_numeric(df_susi[col], errors='coerce')


# 1. '국어_표준점수', '수학_표준점수', '선택1_표준점수', '선택2_표준점수' 열의 합계를 '표점합_수능' 열에 저장
df_susi['표점합수능'] = df_susi['국어_표준점수'] + df_susi['수학_표준점수'] + df_susi['선택1_표준점수'] + df_susi['선택2_표준점수']

# 2. '국어_백분위', '수학_백분위', '선택1_백분위', '선택2_백분위'를 사용하여 '백분위_수능' 열 계산
df_susi['백분위수능'] = (df_susi['국어_백분위'] + df_susi['수학_백분위'] + (df_susi['선택1_백분위'] + df_susi['선택2_백분위']) / 2) / 3

# 3. '국어_등급', '수학_등급', '영어_등급', '선택1_등급', '선택2_등급'을 사용하여 '등급_수능' 열 계산
df_susi['등급수능'] = (df_susi['국어_등급'] + df_susi['수학_등급'] + df_susi['영어_등급'] + (df_susi['선택1_등급'] + df_susi['선택2_등급']) / 2) / 4



# 4.'표점합수능' 열에서 NaN 값을 0으로 대체하고 정수형으로 변환
df_susi['표점합수능'] = df_susi['표점합수능'].fillna(0).astype(int)

# 5. 백분위수능 값을 소수점 첫째 자리까지 표시
df_susi['백분위수능'] = round(df_susi['백분위수능'], 1)

# 6. 등급수능을 소수점 첫째 자리까지 표시
df_susi['등급수능'] = round(df_susi['등급수능'], 1)

# 7.새로운 열 생성 및 값 연결
df_susi['이름_대학_모집단위_최종_표점합수능_백분위수능_등급수능'] = df_susi['이름'].astype(str) + ' ' + \
    df_susi['대학'].astype(str) + ' ' + df_susi['모집단위'].astype(str) + ' ' + \
    df_susi['최종'].astype(str) + '/' + df_susi['표점합수능'].astype(str) + '점' + ' ' + \
    df_susi['백분위수능'].astype(str) + '%' + ' ' + df_susi['등급수능'].astype(str) + '등급'

# 8. 새로운 열 생성 및 값 연결
df_susi['대학_모집단위_최종_표점합수능_백분위수능_등급수능'] = df_susi['대학'].astype(str) + ' ' + \
    df_susi['모집단위'].astype(str) + ' ' + \
    df_susi['최종'].astype(str) + '/' + df_susi['표점합수능'].astype(str) + '점' + ' ' + \
    df_susi['백분위수능'].astype(str) + '%' + ' ' + df_susi['등급수능'].astype(str) + '등급'


# 결과 DataFrame 출력
print(df_susi.tail())


# 새로운 경로와 파일명으로 엑셀 파일 저장
new_path = '/content/drive/MyDrive/python/habbul/'
df_susi.to_excel(f'{new_path}eunpyeong_2023_habbul.xlsx', index=False)